In [14]:
from __future__ import annotations
import json
import os
import re
import statistics
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime as dt_datetime
from dotenv import load_dotenv

## Carrega variáveis do .env

In [15]:
load_dotenv()

True

## Categorizar Log

In [16]:

# ---------------------------------------------------------------------------
# Linha base de todo evento estruturado
# ---------------------------------------------------------------------------
LINHAS_BASE = re.compile(
    r"^(?P<ts>\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}) "
    r"(?P<level>INFO|WARN|ERROR|DEBUG) "
    r"(?P<component>[\w$.]+): "
    r"(?P<msg>.*)$"
)

# Formato real do log4j do Spark: yy/MM/dd (NAO dd/MM/yy).
FORMATO_TS = "%y/%m/%d %H:%M:%S"

# ---------------------------------------------------------------------------
# Mascara de dados sensiveis - host/IP e paths s3a://
# ---------------------------------------------------------------------------
HOST_PATTERN = re.compile(r"\[?([0-9a-fA-F:\.]{7,})\]?(?::\d+)?")
PATH_PATTERN = re.compile(r"s3a?://[^\s,)\"]+")

# Prefixo do diretorio temporario onde o Spark descompacta os --py-files.
# Precisa sair antes de agrupar call sites, senao cada execucao (UUID novo)
# vira um call site diferente e o agrupamento nao acumula nada.
TMP_SPARK_PATTERN = re.compile(r"/tmp/spark-[0-9a-f\-]{8,}/")


class Sanitizador:
    def __init__(self):
        self.host_map: dict[str, str] = {}
        self.path_map: dict[str, str] = {}

    def mask_host(self, host: str) -> str:
        m = re.match(r"\[?([0-9a-fA-F:\.]+?)\]?(?::\d+)?$", host)
        normalized = m.group(1) if m else host
        if normalized not in self.host_map:
            self.host_map[normalized] = f"host_{len(self.host_map) + 1}"
        return self.host_map[normalized]

    def mask_paths(self, text: str) -> str:
        def _replace(m):
            raw = m.group(0)
            if raw not in self.path_map:
                self.path_map[raw] = f"path_{len(self.path_map) + 1}"
            return self.path_map[raw]
        return PATH_PATTERN.sub(_replace, text)

    def mask_text(self, text: str) -> str:
        text = self.mask_paths(text)

        def _replace_host(m):
            return f"[{self.mask_host(m.group(1))}]"
        return HOST_PATTERN.sub(_replace_host, text)


# ---------------------------------------------------------------------------
# Call site
# ---------------------------------------------------------------------------
CALL_SITE_PATTERN = re.compile(r"^(?P<metodo>\S+) at (?P<arquivo>[^:]+):(?P<linha>\d+)$")

# Metodos que o Spark rotula como acao mas que sao internos do Delta/engine,
# nao codigo do usuario. Separar isso evita acusar "acao em laco" onde o laco
# e do proprio Delta.
METODOS_INTERNOS = {
    "getHistory",
    "$anonfun$recordDeltaOperationInternal$1",
    "$anonfun$withThreadLocalCaptured$1",
}


def normalizar_acao(action: str) -> str:
    """Remove o diretorio temporario volatil do call site."""
    return TMP_SPARK_PATTERN.sub("", action.strip())


def extrair_call_site(action: str) -> dict | None:
    """
    Decompoe o call site que o Spark registra em todo job (ex: "count at
    TransformacaoApolice.scala:145") em metodo/arquivo/linha do codigo do
    usuario. Retorna None quando o formato nao bate (ex: acoes sem call
    site legivel, comuns em alguns planos gerados internamente).
    """
    m = CALL_SITE_PATTERN.match(normalizar_acao(action))
    if not m:
        return None
    metodo = m.group("metodo")
    return {
        "metodo": metodo,
        "arquivo": m.group("arquivo"),
        "linha": int(m.group("linha")),
        "interno": metodo in METODOS_INTERNOS,
    }


PADROES_DE_EVENTOS = {
    "job_start": re.compile(r"^Got job (?P<job_id>\d+) \((?P<action>.*?)\) with (?P<partitions>\d+) output partitions$"),
    "job_finish": re.compile(r"^Job (?P<job_id>\d+) finished: .*?, took (?P<duration_s>[\d.]+) s$"),
    "stage_submit": re.compile(r"^Submitting (?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+)"),
    "stage_finish": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) finished in (?P<duration_s>[\d.]+) s$"),
    "taskset_removed": re.compile(r"^Removed TaskSet (?P<stage_id>\d+)\.(?P<attempt>\d+), whose tasks have all completed"),
    "stage_failed": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) failed"),
    "task_start": re.compile(
        r"^Starting task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"\((?P<host>[^,]+), executor (?P<executor>\d+)"
    ),
    "task_finish": re.compile(
        r"^Finished task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"in (?P<duration_ms>\d+) ms on (?P<host>[^\s(]+) \(executor (?P<executor>\d+)\)"
    ),
    "broadcast_stored": re.compile(
        r"^Block broadcast_(?P<broadcast_id>\d+)(?:_piece\d+)? stored as (?:values|bytes) in memory "
        r"\(estimated size (?P<size_val>[\d.]+) (?P<size_unit>\w+),(?: actual size: [\d.]+ \w+,)? "
        r"free (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "block_added": re.compile(
        r"^Added broadcast_(?P<broadcast_id>\d+)_piece\d+ in memory on "
        r"(?P<host>\[[0-9a-fA-F:\.]+\]:\d+) "
        r"\(size: (?P<size_val>[\d.]+) (?P<size_unit>\w+), free: (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "shuffle_map_output_request": re.compile(r"^Asked to send map output locations for shuffle (?P<shuffle_id>\d+)$"),

    # -----------------------------------------------------------------------
    # NOVO 1 - Window sem partitionBy.
    # O Spark move TODO o dataset para uma unica particao. E o sinal de maior
    # relacao custo/beneficio do log: uma linha de WARN por operador.
    # -----------------------------------------------------------------------
    "window_sem_particao": re.compile(r"^No Partition Defined for Window operation"),

    # -----------------------------------------------------------------------
    # NOVO 2 - numero real de tasks submetidas por stage, com a descricao do
    # RDD. Cruzado com a duracao mediana da task, separa paralelismo legitimo
    # de overhead puro de agendamento.
    # -----------------------------------------------------------------------
    "stage_tasks_submitted": re.compile(
        r"^Submitting (?P<num_tasks>\d+) missing tasks from "
        r"(?P<stage>ShuffleMapStage|ResultStage) (?P<stage_id>\d+) \((?P<rdd_desc>.*?)\)"
    ),

    # -----------------------------------------------------------------------
    # NOVO 3 - substitui o antigo "shuffle_partition_advisory", capturando
    # tambem o alvo efetivo e o tamanho minimo. Quando actual << advisory, o
    # AQE esta fatiando o shuffle em particoes muito menores do que o alvo.
    # O shuffle_id pode vir com varios ids ("shuffle(0, 1)").
    # -----------------------------------------------------------------------
    "aqe_coalesce_target": re.compile(
        r"^For shuffle\((?P<shuffle_id>[\d,\s]+)\), advisory target size: (?P<advisory_bytes>\d+), "
        r"actual target size (?P<actual_bytes>\d+), minimum partition size: (?P<min_bytes>\d+)$"
    ),

    # -----------------------------------------------------------------------
    # NOVO 4 - planejamento de leitura. num_files alto com max_split_bytes no
    # piso (= open cost) indica small files no source.
    # -----------------------------------------------------------------------
    "scan_bin_packing": re.compile(
        r"^Planning scan with bin packing, max size: (?P<max_split_bytes>\d+) bytes, "
        r"open cost is considered as scanning (?P<open_cost_bytes>\d+) bytes, "
        r"number of split files: (?P<num_files>\d+), prefetch: (?P<prefetch>\w+)$"
    ),

    # -----------------------------------------------------------------------
    # NOVO 5 - teto efetivo de executores e cores por executor: sem isso nao
    # da para calcular slots nem ocupacao do cluster.
    # -----------------------------------------------------------------------
    "max_executors": re.compile(r"^Using effectiveMaxExecutors = (?P<max_executors>\d+)"),
    "resource_profile": re.compile(r"^Default ResourceProfile created, executor resources: (?P<recursos>.*)$"),

    # -----------------------------------------------------------------------
    # NOVO 6 - Delta: leitura de snapshot/checkpoint por tabela.
    # -----------------------------------------------------------------------
    "delta_snapshot": re.compile(
        r"^Loading version (?P<version>\d+) starting from checkpoint version (?P<checkpoint>\d+)\.?$"
    ),

    # -----------------------------------------------------------------------
    # NOVO 7 - contexto da aplicacao, para o cabecalho do relatorio.
    # -----------------------------------------------------------------------
    "app_name": re.compile(r"^Submitted application: (?P<app_name>.+)$"),
    "spark_version": re.compile(r"^Running Spark version (?P<spark_version>\S+)$"),
    "codegen": re.compile(r"^Code generated in (?P<duration_ms>[\d.]+) ms$"),

    "executor_backlog_request": re.compile(r"^Requesting (?P<num_requested>\d+) new executors because tasks are backlogged"),
    "executor_registered": re.compile(r"^New executor (?P<executor>\d+) has registered \(new total is (?P<total>\d+)\)$"),
    "executor_not_found": re.compile(r"^No executor found for (?P<host>[0-9a-fA-F:\.]+)$"),
    "executor_lost": re.compile(r"^Lost executor (?P<executor>\d+) on (?P<host>[^:]+): (?P<reason>.+)$"),
    "app_final_status": re.compile(r"^SparkContext is stopping with exitCode (?P<exit_code>\d+)\.?$"),
}


@dataclass
class LogEvent:
    ts: str
    level: str
    component: str
    event_type: str
    data: dict = field(default_factory=dict)


def parse_log(linhas, sanitizador: Sanitizador | None = None) -> list[LogEvent]:
    if sanitizador is None:
        sanitizador = Sanitizador()

    eventos: list[LogEvent] = []
    pendente_stack_trace: LogEvent | None = None

    for linha_bruta in linhas:
        linha = linha_bruta.rstrip("\n")
        m = LINHAS_BASE.match(linha.strip())

        if not m:
            if pendente_stack_trace is not None and linha.strip():
                pendente_stack_trace.data.setdefault("stack_trace", [])
                pendente_stack_trace.data["stack_trace"].append(
                    sanitizador.mask_text(linha.strip())
                )
            continue

        ts, level, component, msg = m.group("ts", "level", "component", "msg")
        pendente_stack_trace = None

        matched = False
        for event_type, pattern in PADROES_DE_EVENTOS.items():
            em = pattern.match(msg)
            if em:
                data = em.groupdict()
                if "host" in data and data["host"]:
                    data["host_masked"] = sanitizador.mask_host(data.pop("host"))
                if event_type == "job_start" and data.get("action"):
                    data["action"] = normalizar_acao(data["action"])
                    call_site = extrair_call_site(data["action"])
                    if call_site:
                        data["call_site"] = call_site
                if event_type == "stage_tasks_submitted" and data.get("rdd_desc"):
                    data["rdd_desc"] = normalizar_acao(data["rdd_desc"])
                evento = LogEvent(ts, level, component, event_type, data)
                eventos.append(evento)
                matched = True
                break

        if not matched and level in ("WARN", "ERROR"):
            evento = LogEvent(ts, level, component, "raw_warn_error",
                              {"msg": sanitizador.mask_text(msg)})
            eventos.append(evento)
            pendente_stack_trace = evento

    return eventos


def mapear_stage_para_origem(eventos: list[LogEvent]) -> dict[str, dict]:
    """
    Liga cada stage_id ao job (e, portanto, a linha do script) que o
    originou. Usa a ORDEM CRONOLOGICA do log: o DAGScheduler do Spark
    processa um job por vez, entao todo 'stage_submit' pertence ao job
    mais recente iniciado antes dele.

    PREMISSA (documentar na METODOLOGIA.MD): isso assume execucao
    sequencial de jobs no driver. Em cenarios de jobs concorrentes
    (multiplas threads disparando actions em paralelo, scheduler FAIR
    com pools), essa ligacao pode ficar imprecisa e deve ser tratada
    como "melhor esforco", nao como verdade absoluta.
    """
    mapa: dict[str, dict] = {}
    job_atual: dict | None = None

    for e in eventos:
        if e.event_type == "job_start":
            job_atual = {
                "job_id": e.data.get("job_id"),
                "acao": e.data.get("action"),
                "call_site": e.data.get("call_site"),
            }
        elif e.event_type == "stage_submit" and job_atual is not None:
            mapa.setdefault(e.data["stage_id"], job_atual)

    return mapa


def agregado_por_stage(eventos: list[LogEvent]) -> list[dict]:
    tasks: dict[str, dict] = {}
    for e in eventos:
        if e.event_type in ("task_start", "task_finish"):
            tasks.setdefault(e.data["tid"], {}).update(e.data)

    by_stage: dict[str, list[dict]] = defaultdict(list)
    for t in tasks.values():
        if "duration_ms" in t and "stage_id" in t:
            stage_id = t["stage_id"].split(".")[0]
            by_stage[stage_id].append(t)

    retries_por_stage: dict[str, int] = defaultdict(int)
    for t in tasks.values():
        task_id = t.get("task_id", "")
        stage_id_t = t.get("stage_id", "").split(".")[0]
        partes = task_id.split(".")
        if len(partes) == 2 and partes[1].isdigit() and int(partes[1]) > 0:
            retries_por_stage[stage_id_t] += 1

    origem_por_stage = mapear_stage_para_origem(eventos)

    # NOVO: tasks submetidas + descricao do RDD, por stage.
    submetidas: dict[str, dict] = {}
    for e in eventos:
        if e.event_type == "stage_tasks_submitted":
            submetidas[e.data["stage_id"]] = {
                "num_tasks_submetidas": int(e.data["num_tasks"]),
                "rdd_desc": e.data.get("rdd_desc"),
                "tipo_stage": e.data.get("stage"),
            }

    stage_meta = {}
    for e in eventos:
        if e.event_type == "stage_finish":
            stage_meta[e.data["stage_id"]] = {"duration_s": float(e.data["duration_s"]), "fonte": "stage_finish"}
        elif e.event_type == "taskset_removed" and e.data["stage_id"] not in stage_meta:
            stage_meta[e.data["stage_id"]] = {"duration_s": None, "fonte": "taskset_removed (sem duracao precisa)"}

    summary = []
    for stage_id, task_list in by_stage.items():
        durations = [int(t["duration_ms"]) for t in task_list]
        per_executor = defaultdict(list)
        for t in task_list:
            per_executor[t.get("executor", "?")].append(int(t["duration_ms"]))

        durations_sorted = sorted(durations)
        p95_idx = max(0, int(len(durations_sorted) * 0.95) - 1)
        median = statistics.median(durations)
        sub = submetidas.get(stage_id, {})

        summary.append({
            "stage_id": stage_id,
            "tipo_stage": sub.get("tipo_stage"),
            "rdd_desc": sub.get("rdd_desc"),
            "num_tasks": len(task_list),
            "num_tasks_submetidas": sub.get("num_tasks_submetidas"),
            "duration_s": stage_meta.get(stage_id, {}).get("duration_s"),
            "duration_fonte": stage_meta.get(stage_id, {}).get("fonte"),
            "task_duration_ms": {
                "min": min(durations),
                "max": max(durations),
                "mean": round(statistics.mean(durations), 1),
                "median": median,
                "p95": durations_sorted[p95_idx],
                "total": sum(durations),
            },
            "tasks_per_executor": {ex: len(v) for ex, v in per_executor.items()},
            "executor_duration_ms": {
                ex: {"mean": round(statistics.mean(v), 1), "total": sum(v)}
                for ex, v in per_executor.items()
            },
            "skew_ratio": round(max(durations) / median, 2) if median > 0 else None,
            "task_retries": retries_por_stage.get(stage_id, 0),
            "origem": origem_por_stage.get(stage_id),
        })

    return sorted(summary, key=lambda s: int(s["stage_id"]))


# ---------------------------------------------------------------------------
# NOVO: agregacao por call site.
# O mesmo arquivo:linha aparecendo em N jobs e a assinatura de acao dentro de
# laco no driver - cada iteracao reexecuta o plano inteiro.
# ---------------------------------------------------------------------------
def agregado_por_call_site(eventos: list[LogEvent]) -> list[dict]:
    jobs: dict[str, dict] = {}
    for e in eventos:
        if e.event_type == "job_start":
            jobs.setdefault(e.data["job_id"], {}).update({
                "acao": e.data.get("action"),
                "call_site": e.data.get("call_site"),
                "output_partitions": int(e.data.get("partitions", 0)),
            })
        elif e.event_type == "job_finish":
            jobs.setdefault(e.data["job_id"], {})["duration_s"] = float(e.data["duration_s"])

    agrupado: dict[str, dict] = {}
    for job_id, j in jobs.items():
        chave = j.get("acao") or "(sem call site)"
        alvo = agrupado.setdefault(chave, {
            "acao": chave,
            "call_site": j.get("call_site"),
            "num_jobs": 0,
            "duracao_total_s": 0.0,
            "output_partitions_total": 0,
            "job_ids": [],
        })
        alvo["num_jobs"] += 1
        alvo["duracao_total_s"] += j.get("duration_s", 0.0)
        alvo["output_partitions_total"] += j.get("output_partitions", 0)
        alvo["job_ids"].append(job_id)

    saida = []
    for v in agrupado.values():
        v["duracao_total_s"] = round(v["duracao_total_s"], 2)
        v["duracao_media_s"] = round(v["duracao_total_s"] / v["num_jobs"], 3)
        saida.append(v)
    return sorted(saida, key=lambda x: -x["duracao_total_s"])


# ---------------------------------------------------------------------------
# NOVO: capacidade e ocupacao do cluster.
# ---------------------------------------------------------------------------
def capacidade_cluster(eventos: list[LogEvent], stage_summary: list[dict],
                       duracao_total_s: float | None) -> dict:
    max_exec = None
    cores = None
    for e in eventos:
        if e.event_type == "max_executors":
            max_exec = int(e.data["max_executors"])
        elif e.event_type == "resource_profile":
            m = re.search(r"cores\s*->\s*name:\s*cores,\s*amount:\s*(\d+)", e.data.get("recursos", ""))
            if m:
                cores = int(m.group(1))

    executores_vistos = set()
    for s in stage_summary:
        executores_vistos.update(s["tasks_per_executor"].keys())
    executores_vistos.discard("?")

    tempo_task_s = sum(s["task_duration_ms"]["total"] for s in stage_summary) / 1000
    slots = (max_exec * cores) if (max_exec and cores) else None
    capacidade_s = (slots * duracao_total_s) if (slots and duracao_total_s) else None

    return {
        "max_executors": max_exec,
        "cores_por_executor": cores,
        "slots": slots,
        "executores_com_tasks": sorted(executores_vistos, key=lambda x: int(x) if x.isdigit() else 0),
        "tempo_total_task_s": round(tempo_task_s, 1),
        "capacidade_total_s": round(capacidade_s, 1) if capacidade_s else None,
        "ocupacao_pct": round(100 * tempo_task_s / capacidade_s, 1) if capacidade_s else None,
        "fonte": "slots = effectiveMaxExecutors x cores por executor; ocupacao = tempo de task / capacidade",
    }


# ---------------------------------------------------------------------------
# NOVO: motor de deteccao de oportunidades.
# Cada regra devolve um achado com severidade, evidencia e acao sugerida.
# Os limiares ficam centralizados aqui para serem calibrados sem mexer nas
# regras.
# ---------------------------------------------------------------------------
LIMIARES = {
    "stage_explodido_tasks": 500,       # tasks num unico stage
    "stage_explodido_mediana_ms": 100,  # com mediana abaixo disso = overhead
    "aqe_razao_alvo": 8,                # advisory / actual acima disso = alerta
    "acao_repetida_min_jobs": 5,        # mesmo call site em N+ jobs = laco
    "skew_ratio": 3.0,
    "skew_min_tasks": 8,
    "skew_min_max_ms": 5000,            # razao alta em stage curto e ruido
    "stage_1_task_pct": 40,             # % de stages com uma unica task
    "small_files_min": 50,              # arquivos num scan no piso de split
    "ocupacao_baixa_pct": 60,
}


def _achado(regra, severidade, titulo, evidencia, acao):
    return {
        "regra": regra,
        "severidade": severidade,
        "titulo": titulo,
        "evidencia": evidencia,
        "acao_sugerida": acao,
    }


def detectar_oportunidades(eventos: list[LogEvent], stage_summary: list[dict],
                           call_sites: list[dict], capacidade: dict) -> list[dict]:
    achados = []

    # 1. Window sem partitionBy
    janelas = [e for e in eventos if e.event_type == "window_sem_particao"]
    if janelas:
        achados.append(_achado(
            "window_sem_particao", "ALTA",
            "Window function sem partitionBy",
            {"ocorrencias": len(janelas), "primeiro_ts": janelas[0].ts, "ultimo_ts": janelas[-1].ts},
            "Todo o dataset e movido para uma unica particao. Adicionar partitionBy "
            "na chave de negocio; se a numeracao precisa ser global, trocar por "
            "zipWithIndex/monotonically_increasing_id.",
        ))

    # 2. Stage explodido: muitas tasks, cada uma trivial
    for s in stage_summary:
        n = s["num_tasks_submetidas"] or s["num_tasks"]
        if n >= LIMIARES["stage_explodido_tasks"] and \
           s["task_duration_ms"]["median"] < LIMIARES["stage_explodido_mediana_ms"]:
            achados.append(_achado(
                "stage_explodido", "ALTA",
                f"Stage {s['stage_id']} com {n} tasks triviais",
                {
                    "stage_id": s["stage_id"],
                    "num_tasks": n,
                    "mediana_ms": s["task_duration_ms"]["median"],
                    "tempo_total_task_s": round(s["task_duration_ms"]["total"] / 1000, 1),
                    "rdd_desc": s["rdd_desc"],
                },
                "Particoes pequenas demais: quase todo o tempo e overhead de "
                "agendamento. Revisar coalesce/repartition antes do write e o "
                "minPartitionSize do AQE.",
            ))

    # 3. AQE coalescendo para muito abaixo do alvo
    alvos = [e for e in eventos if e.event_type == "aqe_coalesce_target"]
    suspeitos = [
        e for e in alvos
        if int(e.data["actual_bytes"]) > 0
        and int(e.data["advisory_bytes"]) / int(e.data["actual_bytes"]) >= LIMIARES["aqe_razao_alvo"]
    ]
    if suspeitos:
        exemplo = suspeitos[0].data
        achados.append(_achado(
            "aqe_alvo_reduzido", "MEDIA",
            "AQE coalescendo shuffle muito abaixo do advisory",
            {
                "ocorrencias": len(suspeitos),
                "advisory_bytes": int(exemplo["advisory_bytes"]),
                "actual_bytes": int(exemplo["actual_bytes"]),
                "min_partition_bytes": int(exemplo["min_bytes"]),
                "razao": round(int(exemplo["advisory_bytes"]) / int(exemplo["actual_bytes"]), 1),
            },
            "Ajustar spark.sql.adaptive.coalescePartitions.minPartitionSize e "
            "avaliar coalescePartitions.parallelismFirst=false.",
        ))

    # 4. Acao repetida = laco no driver
    for cs in call_sites:
        if cs["num_jobs"] >= LIMIARES["acao_repetida_min_jobs"]:
            interno = bool(cs.get("call_site") and cs["call_site"].get("interno"))
            achados.append(_achado(
                "acao_repetida", "MEDIA" if not interno else "BAIXA",
                f"{cs['num_jobs']} jobs disparados por {cs['acao'][:70]}",
                {
                    "acao": cs["acao"],
                    "num_jobs": cs["num_jobs"],
                    "duracao_total_s": cs["duracao_total_s"],
                    "interno_do_engine": interno,
                },
                "Acao dentro de laco: cada iteracao reexecuta o plano. Vetorizar "
                "em join/agregacao unica, ou ao menos cachear o DataFrame base."
                if not interno else
                "Chamada interna do Delta/engine repetida (ex: history/metadata). "
                "Cachear o resultado no driver em vez de reconsultar.",
            ))

    # 5. Skew de task
    for s in stage_summary:
        # O piso absoluto e essencial: sem ele, um stage de 9 tasks com
        # mediana de 100 ms e um outlier de 3 s vira "skew 29x" - razao
        # enorme, impacto nenhum.
        if s["skew_ratio"] and s["skew_ratio"] >= LIMIARES["skew_ratio"] \
           and s["num_tasks"] >= LIMIARES["skew_min_tasks"] \
           and s["task_duration_ms"]["max"] >= LIMIARES["skew_min_max_ms"]:
            achados.append(_achado(
                "skew_task", "MEDIA",
                f"Stage {s['stage_id']} com distribuicao desbalanceada",
                {
                    "stage_id": s["stage_id"],
                    "skew_ratio": s["skew_ratio"],
                    "max_ms": s["task_duration_ms"]["max"],
                    "median_ms": s["task_duration_ms"]["median"],
                    "num_tasks": s["num_tasks"],
                },
                "Chave de join/particionamento concentrada. Avaliar salting ou "
                "skewJoin do AQE.",
            ))

    # 6. Fragmentacao: maioria dos stages com 1 task
    if stage_summary:
        um_task = [s for s in stage_summary if s["num_tasks"] == 1]
        pct = 100 * len(um_task) / len(stage_summary)
        if pct >= LIMIARES["stage_1_task_pct"]:
            achados.append(_achado(
                "fragmentacao_stages", "BAIXA",
                "Maioria dos stages roda com uma unica task",
                {"stages_com_1_task": len(um_task), "total_stages": len(stage_summary), "pct": round(pct, 1)},
                "Pipeline fragmentado em muitas acoes pequenas: o custo e latencia "
                "de driver, nao processamento. Consolidar transformacoes.",
            ))

    # 7. Small files no scan
    scans = [e for e in eventos if e.event_type == "scan_bin_packing"]
    small = [
        e for e in scans
        if int(e.data["num_files"]) >= LIMIARES["small_files_min"]
        and int(e.data["max_split_bytes"]) <= int(e.data["open_cost_bytes"])
    ]
    if small:
        achados.append(_achado(
            "small_files", "MEDIA",
            "Scan com muitos arquivos pequenos",
            {"ocorrencias": len(small),
             "max_arquivos_em_um_scan": max(int(e.data["num_files"]) for e in small)},
            "Custo de abertura domina a leitura. Rodar OPTIMIZE/compactacao na "
            "tabela de origem.",
        ))

    # 8. Ocupacao baixa do cluster
    if capacidade.get("ocupacao_pct") is not None and \
       capacidade["ocupacao_pct"] < LIMIARES["ocupacao_baixa_pct"]:
        achados.append(_achado(
            "ocupacao_baixa", "MEDIA",
            "Cluster ocioso em boa parte da execucao",
            {k: capacidade[k] for k in ("slots", "tempo_total_task_s", "capacidade_total_s", "ocupacao_pct")},
            "Tempo gasto fora de task (planejamento, metadado, commit). Reduzir "
            "numero de acoes e conferir o teto de executores.",
        ))

    ordem = {"ALTA": 0, "MEDIA": 1, "BAIXA": 2}
    return sorted(achados, key=lambda a: ordem[a["severidade"]])


def resumo_eventos_esparsos(eventos: list[LogEvent]) -> dict:
    contagem = defaultdict(int)
    for e in eventos:
        contagem[e.event_type] += 1
    return dict(sorted(contagem.items(), key=lambda kv: -kv[1]))


def contexto_aplicacao(eventos: list[LogEvent]) -> dict:
    ctx = {"app_name": None, "spark_version": None, "tabelas_delta": []}
    tabelas = []
    for e in eventos:
        if e.event_type == "app_name":
            ctx["app_name"] = e.data["app_name"]
        elif e.event_type == "spark_version":
            ctx["spark_version"] = e.data["spark_version"]
        elif e.event_type == "delta_snapshot":
            tabelas.append({"version": int(e.data["version"]), "checkpoint": int(e.data["checkpoint"])})
    ctx["snapshots_delta_carregados"] = len(tabelas)
    return ctx


def eventos_de_falha(eventos: list[LogEvent]) -> dict:
    stage_failures = [asdict(e) for e in eventos if e.event_type == "stage_failed"]
    executor_lost = [asdict(e) for e in eventos if e.event_type == "executor_lost"]
    return {
        "stage_failures": {"total": len(stage_failures), "detalhes": stage_failures},
        "executor_lost": {"total": len(executor_lost), "detalhes": executor_lost},
    }


def duracao_total_execucao(eventos: list[LogEvent]) -> dict:
    timestamps = []
    for e in eventos:
        try:
            # CORRECAO: o log4j do Spark grava yy/MM/dd, nao dd/MM/yy. Com o
            # formato antigo a data era lida invertida e a duracao estourava
            # em qualquer job que atravessasse a meia-noite.
            timestamps.append(dt_datetime.strptime(e.ts, FORMATO_TS))
        except ValueError:
            continue

    if not timestamps:
        return {"duration_s": None, "fonte": "nenhum timestamp pode ser interpretado"}

    inicio, fim = min(timestamps), max(timestamps)
    return {
        "duration_s": (fim - inicio).total_seconds(),
        "inicio": inicio.strftime(FORMATO_TS),
        "fim": fim.strftime(FORMATO_TS),
        "fonte": "diferenca entre o primeiro e o ultimo timestamp do log",
    }


def metricas_nao_disponiveis() -> dict:
    return {
        "gc_time_ratio": {
            "valor": None,
            "motivo": "Log em nivel INFO padrao, sem eventos de GC verboso (flag -verbose:gc nao habilitada).",
        },
        "shuffle_read_write_bytes": {
            "valor": None,
            "motivo": "Log expoe apenas sinal indireto de shuffle (advisory/actual target size), nao bytes lidos/escritos reais.",
        },
        "spill_memoria_disco": {
            "valor": None,
            "motivo": "Nao ha eventos de spill no log neste nivel de verbosidade.",
        },
        "result_size_driver": {
            "valor": None,
            "motivo": "Result Size por task so existe no event log (SparkListenerTaskEnd). "
                      "Sem ele, o risco de collect e estimado por numero de particoes, nao por bytes.",
        },
    }


if __name__ == "__main__":
    path = os.getenv("LOG_REP_FAT_INTERROMPIDO")

    if not path:
        raise ValueError("A variavel nao foi encontrada no arquivo .env.")

    if not os.path.isfile(path):
        raise FileNotFoundError("Arquivo nao encontrado!")

    sanitizador = Sanitizador()

    with open(path, "r", encoding="utf-8") as f:
        parsed = parse_log(f, sanitizador)

    stage_summary = agregado_por_stage(parsed)
    call_sites = agregado_por_call_site(parsed)
    duracao_total = duracao_total_execucao(parsed)
    capacidade = capacidade_cluster(parsed, stage_summary, duracao_total.get("duration_s"))
    oportunidades = detectar_oportunidades(parsed, stage_summary, call_sites, capacidade)

    raw_eventos = [asdict(e) for e in parsed if e.event_type == "raw_warn_error"]
    app_status = [asdict(e) for e in parsed if e.event_type == "app_final_status"]
    falhas = eventos_de_falha(parsed)

    output = {
        "contexto": contexto_aplicacao(parsed),
        "duracao_total_execucao": duracao_total,
        "capacidade_cluster": capacidade,
        "oportunidades": oportunidades,
        "call_sites": call_sites,
        "stages": stage_summary,
        "task_retries_total": sum(s["task_retries"] for s in stage_summary),
        "stage_failures": falhas["stage_failures"],
        "executor_lost": falhas["executor_lost"],
        "eventos_esparsos": resumo_eventos_esparsos(parsed),
        "metricas_nao_disponiveis": metricas_nao_disponiveis(),
        "status_final": app_status,
        "raw_warn_error": raw_eventos,
    }

    with open("LOG_REP_FAT_INTERROMPIDO.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"Aplicacao : {output['contexto']['app_name']}")
    print(f"Duracao   : {duracao_total['duration_s']} s")
    print(f"Ocupacao  : {capacidade['ocupacao_pct']}% de {capacidade['slots']} slots")
    print(f"Achados   : {len(oportunidades)}")
    for a in oportunidades:
        print(f"  [{a['severidade']:<5}] {a['titulo']}")
    print("\nArquivo 'LOG_REP_FAT_INTERROMPIDO.json' gerado com sucesso.")
    print(f"Hosts mascarados: {len(sanitizador.host_map)} | Paths mascarados: {len(sanitizador.path_map)}")

Aplicacao : REP_FATURAMENTO_ATUAL_DELTA
Duracao   : 64.0 s
Ocupacao  : 0.9% de 80 slots
Achados   : 1
  [MEDIA] Cluster ocioso em boa parte da execucao

Arquivo 'LOG_REP_FAT_INTERROMPIDO.json' gerado com sucesso.
Hosts mascarados: 6 | Paths mascarados: 5
